# 3. Modelos benchmark: selección, reentrenamiento y artefactos

Del universo de 140 combinaciones del Entregable 2, el tablero no carga todas: carga la mejor
combinación de cada uno de los siete modelos de clasificación, elegida con un criterio estadístico
(guía, sección 4). Este capítulo documenta la selección, el reentrenamiento con todo el periodo de
entrenamiento, la serialización con sus metadatos y la prueba de ida y vuelta, y añade dos
comparadores recientes del estado del arte (LightGBM y CatBoost) evaluados con el mismo protocolo.

In [1]:
import json
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)
RES = Path("../data/resultados")
ART = Path("../artifacts")

## 3.1 Criterio de selección

Para cada modelo se elige la combinación (balanceo × optimizador) con mayor AUC medio en los cuatro
pliegues externos de la validación anidada, que es la métrica de selección del Entregable 2. Los
empates exactos se resuelven con el Brier medio (capítulo 1). Para la mejor combinación global se
verifica que pertenezca al grupo de finalistas de Friedman-Nemenyi del Entregable 2: su rango medio
entre las 112 combinaciones de clasificación, en 48 bloques mensuales, debe estar a menos de una
diferencia crítica del mejor rango.

In [2]:
sel = pd.read_csv(RES / "seleccion_mejores.csv")
tabla = pd.DataFrame({
    "Modelo": sel["modelo"], "Balanceo": sel["balanceo"], "Optimizador": sel["optimizador"],
    "AUC externo": [f"{m:.4f} ± {d:.4f}" for m, d in zip(sel["auc_media"], sel["auc_de"])],
    "AUC-PR": [f"{m:.4f} ± {d:.4f}" for m, d in zip(sel["auc_pr_media"], sel["auc_pr_de"])],
    "Brier": [f"{m:.4f} ± {d:.4f}" for m, d in zip(sel["brier_media"], sel["brier_de"])],
    "ECE": [f"{m:.4f} ± {d:.4f}" for m, d in zip(sel["ece_media"], sel["ece_de"])],
    "F1": [f"{m:.4f} ± {d:.4f}" for m, d in zip(sel["f1_media"], sel["f1_de"])],
    "Rango Friedman (112)": sel["rango_friedman"].round(2), "Finalista Nemenyi": sel["finalista_nemenyi"],
    "Motivo": sel["motivo"]})
display(tabla)
cd = float(pd.read_parquet(RES / "comp_friedman_global.parquet")["CD"].iloc[0])
display(Markdown(f"La diferencia crítica de Nemenyi entre 112 combinaciones y 48 bloques es {cd:.2f} rangos. "
                 f"XGBoost sin balanceo con Random Search tiene el mejor rango ({sel['rango_friedman'].iloc[0]:.2f}) y, "
                 "por definición, pertenece al grupo de finalistas; Random Forest y la SVM también quedan a menos de una "
                 "diferencia crítica, mientras que la regresión logística, k-NN, Decision Tree y Naive Bayes quedan fuera."))

,Modelo,Balanceo,Optimizador,AUC externo,AUC-PR,Brier,ECE,F1,Rango Friedman (112),Finalista Nemenyi,Motivo
0,XGBoost,Sin balanceo,Random Search,0.7867 ± 0.0027,0.5632 ± 0.0125,0.1615 ± 0.0041,0.0115 ± 0.0037,0.5843 ± 0.0079,5.62,True,Mayor AUC externo de sus 16 combinaciones (0.7867 ± 0.0027; +0.0003 sobre la segunda); mejor global y finalista de F...
1,Random Forest,Sin balanceo,Random Search,0.7852 ± 0.0026,0.5602 ± 0.0135,0.1619 ± 0.0039,0.0086 ± 0.0013,0.5838 ± 0.0078,13.38,True,Mayor AUC externo de sus 16 combinaciones (0.7852 ± 0.0026; +0.0000 sobre la segunda)
2,SVM,Sin balanceo,Bayesiana (TPE),0.7844 ± 0.0023,0.5573 ± 0.0128,0.1634 ± 0.0037,0.0298 ± 0.0040,0.5828 ± 0.0079,19.77,True,Mayor AUC externo de sus 16 combinaciones (0.7844 ± 0.0023; +0.0003 sobre la segunda)
3,Regresión logística,Sin balanceo,Grid Search,0.7809 ± 0.0030,0.5503 ± 0.0120,0.1639 ± 0.0041,0.0214 ± 0.0017,0.5803 ± 0.0077,40.09,False,Mayor AUC externo de sus 16 combinaciones (0.7809 ± 0.0030; +0.0000 sobre la segunda)
4,KNN,Sin balanceo,Random Search,0.7751 ± 0.0029,0.5416 ± 0.0144,0.1661 ± 0.0045,0.0273 ± 0.0061,0.5765 ± 0.0079,69.01,False,"Mayor AUC externo de sus 16 combinaciones (0.7751 ± 0.0029; +0.0000 sobre la segunda, empate desempatado por Brier)"
5,Decision Tree,class_weight,Grid Search,0.7736 ± 0.0032,0.5364 ± 0.0128,0.1982 ± 0.0034,0.1647 ± 0.0038,0.5722 ± 0.0076,74.79,False,Mayor AUC externo de sus 16 combinaciones (0.7736 ± 0.0032; +0.0003 sobre la segunda)
6,Naive Bayes,SMOTE,Grid Search,0.7685 ± 0.0045,0.5161 ± 0.0101,0.2317 ± 0.0061,0.1983 ± 0.0058,0.5733 ± 0.0065,87.75,False,Mayor AUC externo de sus 16 combinaciones (0.7685 ± 0.0045; +0.0001 sobre la segunda)


La diferencia crítica de Nemenyi entre 112 combinaciones y 48 bloques es 28.84 rangos. XGBoost sin balanceo con Random Search tiene el mejor rango (5.62) y, por definición, pertenece al grupo de finalistas; Random Forest y la SVM también quedan a menos de una diferencia crítica, mientras que la regresión logística, k-NN, Decision Tree y Naive Bayes quedan fuera.

Las ventajas entre combinaciones de un mismo modelo son pequeñas (columna «Motivo»): la superficie del
AUC es plana cerca del óptimo, como mostró el Entregable 2, y por eso la elección del balanceo pesa más
que la del optimizador. Solo Decision Tree y Naive Bayes se benefician de una técnica de balanceo.

## 3.2 Reentrenamiento y serialización

Cada tubería completa (preprocesamiento, balanceo cuando corresponde y modelo) se reentrena con las
622.244 corredor-semanas del periodo de entrenamiento, con los hiperparámetros que el bucle interno
eligió en el pliegue final, y se serializa con `joblib.dump(..., compress=3)` en `artifacts/<modelo>/`.
Dos decisiones:

* **k-NN** guarda todo su conjunto de referencia. Con 622 mil filas y unas 70 variables el artefacto
  ocuparía unos 350 MB, más que la memoria del plan gratuito de Render, y su número de vecinos
  (k = 301) se eligió para la densidad de una muestra de 40 mil filas. Se usa como referencia la muestra
  del pliegue final, la misma con que se ajustaron sus hiperparámetros.
* **La SVM** (Nyström + LinearSVC) no produce probabilidades. Se envuelve en una clase propia
  (`ConPlatt`) que ajusta el escalado de Platt con puntajes fuera de pliegue, y su umbral de decisión se
  lleva a la escala de probabilidad con esa misma transformación monótona.

Todos los artefactos se guardan con un solo hilo de cómputo para que sus predicciones sean
deterministas.

In [3]:
art = pd.read_csv(RES / "artefactos_resumen.csv")
display(art[["modelo", "filas", "segundos_fit", "ms_por_1000", "mb", "ida_y_vuelta"]].rename(columns={
    "modelo": "Modelo", "filas": "Filas de entrenamiento", "segundos_fit": "Entrenamiento (s)",
    "ms_por_1000": "Inferencia (ms por 1000 filas)", "mb": "Tamaño (MB)", "ida_y_vuelta": "Ida y vuelta idéntica"}).round(3))

,Modelo,Filas de entrenamiento,Entrenamiento (s),Inferencia (ms por 1000 filas),Tamaño (MB),Ida y vuelta idéntica
0,XGBoost,622244,9.274,3.042,0.134,True
1,Random Forest,622244,58.103,7.666,1.680,True
2,SVM,622244,276.099,5.176,0.737,True
3,Regresión logística,622244,10.243,1.231,0.016,True
4,KNN,39826,0.197,74.741,6.848,True
5,Decision Tree,622244,12.343,0.886,0.030,True
6,Naive Bayes,622244,43.701,1.441,0.018,True
7,LightGBM,622244,6.620,12.987,0.510,True
8,CatBoost,622244,9.364,0.831,0.185,True
9,AJ-KL (original),622244,0.000,71.306,11.196,True


## 3.3 Metadatos y prueba de ida y vuelta

Junto a cada artefacto hay un `metadata.json` con las versiones de las librerías, la semilla, los
hiperparámetros finales, el esquema de las 34 columnas de entrada (nombre, tipo y rango válido o
categorías) y las métricas del bucle externo y de la prueba 2025. El tablero construye su formulario de
predicción a partir de ese esquema. La prueba de ida y vuelta predice una muestra fija de 2.000
corredor-semanas de 2025 con el modelo en memoria, lo serializa, lo vuelve a cargar y exige
predicciones idénticas (`np.array_equal`); `tests/test_artefactos.py` repite la comprobación contra las
predicciones guardadas cada vez que se corre la batería de pruebas.

In [4]:
meta = json.loads((ART / "xgboost" / "metadata.json").read_text())
print(json.dumps({k: meta[k] for k in ["modelo", "combinacion", "semilla", "librerias", "hiperparametros",
                                        "filas_entrenamiento", "prueba_ida_y_vuelta"]}, indent=2, ensure_ascii=False))
print("Esquema (primeras 3 columnas):", json.dumps(meta["esquema_entrada"][:3], ensure_ascii=False)[:600], "...")

{
  "modelo": "XGBoost",
  "combinacion": "XGBoost | Sin balanceo | Random Search",
  "semilla": 42,
  "librerias": {
    "python": "3.11.16",
    "plataforma": "macOS-27.0.1-arm64-arm-64bit",
    "numpy": "2.4.6",
    "pandas": "3.0.6",
    "scipy": "1.17.1",
    "scikit-learn": "1.9.1",
    "xgboost": "3.2.0",
    "imbalanced-learn": "0.14.2",
    "joblib": "1.6.0"
  },
  "hiperparametros": {
    "learning_rate": 0.02603982898132,
    "max_depth": 4,
    "min_child_weight": 43.616164034442946,
    "colsample_bytree": 0.6899225066388948,
    "subsample": 0.8919491045532068
  },
  "filas_entrenamiento": 622244,
  "prueba_ida_y_vuelta": {
    "identicas": true,
    "max_dif_abs": 0.0,
    "n": 2000,
    "nota": "repetida después de aligerar el artefacto"
  }
}
Esquema (primeras 3 columnas): [{"nombre": "n_envios8", "tipo": "numérica", "min": 0.5, "max": 1103.0, "p01": 0.5, "p99": 238.5, "mediana": 4.5, "faltantes": 0.0}, {"nombre": "n_alimentos8", "tipo": "numérica", "min": 0.5, "max": 

## 3.4 Desempeño en 2025 con todo el entrenamiento

In [5]:
cols = ["auc", "auc_pr", "brier", "log_loss", "ece", "f1", "precision", "recall", "accuracy"]
t25 = art.set_index("modelo")[[f"{c}_2025" for c in cols]]
t25.columns = cols
display(t25.sort_values("auc", ascending=False).round(4))

,auc,auc_pr,brier,log_loss,ece,f1,precision,recall,accuracy
modelo,,,,,,,,,
CatBoost,0.8011,0.5602,0.1507,0.4554,0.0107,0.5786,0.4655,0.7643,0.7100
LightGBM,0.8011,0.5591,0.1509,0.4563,0.0157,0.5786,0.4645,0.7670,0.7090
AJ-KL (original),0.8007,0.5580,0.1509,0.4553,0.0111,0.5780,0.4695,0.7518,0.7141
XGBoost,0.7999,0.5566,0.1511,0.4561,0.0107,0.5778,0.4583,0.7813,0.7026
SVM,0.7985,0.5523,0.1524,0.4631,0.0267,0.5759,0.4571,0.7781,0.7016
Random Forest,0.7983,0.5545,0.1517,0.4582,0.0151,0.5762,0.4725,0.7380,0.7172
Regresión logística,0.7934,0.5386,0.1537,0.4636,0.0178,0.5696,0.4416,0.8022,0.6843
Decision Tree,0.7927,0.5415,0.1888,0.5473,0.1722,0.5701,0.4492,0.7800,0.6936
KNN,0.7851,0.5264,0.1561,0.4695,0.0203,0.5656,0.4580,0.7394,0.7042


## 3.5 Comparadores recientes del estado del arte

La guía pide comparar el modelo original también con métodos recientes que no se vieron en clase.
LightGBM {cite}`ke2017lightgbm` y CatBoost {cite}`prokhorenkova2018catboost` se evaluaron con
exactamente el mismo protocolo que los 140 modelos y que el modelo original: los mismos pliegues
externos e internos, la misma muestra de 40 mil filas, el mismo preprocesamiento, TPE con 24
evaluaciones y semilla 42, y el AUC interno como criterio (`scripts/16_sota_sipsa.py`).

In [6]:
try:
    import joblib
    filas = []
    for m in ["LightGBM", "CatBoost"]:
        for pl in ["k1", "k2", "k3", "k4", "final"]:
            r = joblib.load(RES / "sota" / f"{m}_{pl}.joblib")["resumen"]
            filas.append({"modelo": m, "pliegue": pl, **{c: r[c] for c in cols}, "t_total_s": r["t_total"]})
    sota = pd.DataFrame(filas)
    ext = sota[sota.pliegue != "final"].groupby("modelo")[cols].agg(["mean", "std"]).round(4)
    display(ext)
    display(sota[sota.pliegue == "final"].set_index("modelo")[cols].round(4))
except FileNotFoundError:
    print("Resultados de LightGBM y CatBoost pendientes.")

auc          auc_pr           brier         log_loss             ece              f1         precision          recall         accuracy        
            mean     std    mean     std    mean     std     mean     std    mean     std    mean     std      mean     std    mean     std     mean     std
modelo                                                                                                                                                      
CatBoost  0.7860  0.0026  0.5623  0.0122  0.1617  0.0040   0.4844  0.0102  0.0089  0.0007  0.5848  0.0072    0.4706  0.0096  0.7724  0.0102   0.6898  0.0069
LightGBM  0.7865  0.0029  0.5630  0.0114  0.1616  0.0042   0.4837  0.0106  0.0108  0.0029  0.5826  0.0095    0.4633  0.0223  0.7878  0.0353   0.6807  0.0156

,auc,auc_pr,brier,log_loss,ece,f1,precision,recall,accuracy
modelo,,,,,,,,,
LightGBM,0.7981,0.5539,0.1517,0.4583,0.0113,0.5777,0.4680,0.7544,0.7127
CatBoost,0.7982,0.5537,0.1515,0.4576,0.0057,0.5777,0.4679,0.7548,0.7126
